# Covariance-model comparisons


Computational scripts save assignment `-1` first, then random assignments `0–99`.
`PAIRING_VIEW` changes only the loaded/displayed results. Full results exclude `-1`
and give each random assignment equal weight after averaging its internal folds.
Mean ± SD are descriptive, not confidence intervals. Missing assignments are an
error; assignment zero and incomplete batches are never substituted for the full result.
Raw signal preparation runs once in a terminal and includes every eligible MEG subject.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
PAIRING_VIEW = 'matched'  # 'matched': -1; 'full': mean ± SD of IDs 0–99.
N_PAIRINGS = 100
PAIRING_SEED = 2026
from vs0_run_pairings import activate, format_command, pairing_context
activate(PAIRING_VIEW, N_PAIRINGS, PAIRING_SEED, display_only=True)
import vs0_load_and_config as shared_data
from vs0_pairing_results import config_view, path_view, show, safe_load, ResultView
ANALYSIS_OUTPUT_ROOT = shared_data.analysis_output_root(ROOT)
DISPLAY_COMPONENTS = 5
SEED = 2026  # Trial/source-control seed, independent of PAIRING_SEED.
BLOCK_SCALING = True
N_COMPONENTS = 10
CONDITION_MODE = 'average'
MAX_GRAM_GIB = 2.
from vs1_coverage_figures import apply_style, MODALITY_COLORS, SETUP_COLORS
apply_style()

MEG_KIND='paired_coverage'
OUTPUT_DIR=ANALYSIS_OUTPUT_ROOT/'covariance'

## Configuration and terminal preparation

In [ ]:
MEG_DIR = shared_data.MEG_DATA_DIR
IEEG_DIR = ROOT/'ieeg_shortWOBS_fs250'
METADATA_CSV = None
PROJECT_PATH = None
SFREQ = 250.
MEG_TMIN = None
MEG_TIMES_FILE = None
CORRECT_MEG_ZSCORE = True
CORRECT_IEEG_X1000 = True
CONDITIONS = [1,2]
CONFIG = dict(data=dict(meg_dir=str(MEG_DIR), ieeg_dir=str(IEEG_DIR),
    metadata_csv=str(METADATA_CSV) if METADATA_CSV else None,
    project_path=str(PROJECT_PATH) if PROJECT_PATH else None,
    meg_coordinate_unit='m', ieeg_coordinate_unit='mm', meg_tmin=MEG_TMIN,
    meg_scaling='channel_zscore' if CORRECT_MEG_ZSCORE else 'none',
    ieeg_scaling='none', ieeg_multiplier=1000. if CORRECT_IEEG_X1000 else 1.,
    meg_times_file=str(MEG_TIMES_FILE) if MEG_TIMES_FILE else None,
    include_all_meg=True, sfreq=SFREQ, conditions=CONDITIONS,
    prepared_cache_dir=str(ROOT/'out'/'prepared_data')),
    output_dir=str(OUTPUT_DIR), block_scaling=BLOCK_SCALING, condition_mode=CONDITION_MODE,
    n_components=N_COMPONENTS, seed=SEED, pairing_seed=PAIRING_SEED,
    n_pairings=N_PAIRINGS, max_gram_gib=MAX_GRAM_GIB)

CONFIG['meg_kind']=MEG_KIND

In [ ]:
# This cell writes configuration and prints a terminal command; no raw signals are prepared here.
CONFIG_FILE = OUTPUT_DIR/'config.json'
CONFIG_FILE.parent.mkdir(parents=True, exist_ok=True)
CONFIG_FILE.write_text(json.dumps(CONFIG, indent=2))
import shlex
print(shlex.join(['python','-u',str(ROOT/'vs0_prepare_data.py'),'--config',str(CONFIG_FILE)]))

## Fit and load all saved results

In [ ]:
RECOMPUTE=False
command=['python','-u',str(ROOT/'vs2_run_cov_models.py'),'--config',str(CONFIG_FILE)]
if RECOMPUTE: command.append('--recompute')
print(format_command(command))
models_view=safe_load(config_view, CONFIG, 'models', PAIRING_VIEW, N_PAIRINGS)

## Variance explained, shared covariance, native matching and fixed-PCA-reference metrics

Each assignment is fitted before aggregation. Reconstruction variance and cross-covariance energy have different denominators and must not be added. Component pair identities remain labels, not averaged indices.

In [ ]:
if models_view is not None:
    show(models_view, max_components=DISPLAY_COMPONENTS, tables=True, arrays=False)

## Time courses

Full scores are aligned to the matched fit from fitted temporal scores. Separate PCA is aligned independently by modality; joint PCA and PLSSVD retain coupled signs and permutations.

In [ ]:
if models_view is not None:
    scores=ResultView(models_view.view,models_view.assignment_ids,{}, {k:v for k,v in models_view.arrays.items() if 'scores' in k}, models_view.times)
    show(scores,max_components=DISPLAY_COMPONENTS)

## Covariance structure, component variances and signed correlations

Every matrix shows its mean and a separate SD panel in full view. SD is across assignment-level matrices. All matrices below use matched-reference component correspondence.

In [ ]:
if models_view is not None:
    structure=ResultView(models_view.view,models_view.assignment_ids,{}, {k:v for k,v in models_view.arrays.items() if 'scores' not in k}, models_view.times)
    show(structure,max_components=DISPLAY_COMPONENTS)

## Within-modality model comparisons

The tables above include all subspace, individually matched component, and full component-correlation metrics, with assignment SD and valid counts. Native scalar spectra/rank metrics retain native order; aligned score/matrix displays use the matched reference.